In [ ]:
!wget -O - https://data.hplt-project.org/three/sorted/deu_Latn.map | wget -x -nH --cut-dirs=2 -i -

In [ ]:
!wget -O - https://data.hplt-project.org/three/sorted/fra_Latn.map | wget -x -nH --cut-dirs=2 -i -

In [ ]:
!wget -O - https://data.hplt-project.org/three/sorted/ita_Latn.map | wget -x -nH --cut-dirs=2 -i -

In [ ]:
!wget -O - https://data.hplt-project.org/three/sorted/swe_Latn.map | wget -x -nH --cut-dirs=2 -i -

In [ ]:
!wget -O - https://data.hplt-project.org/three/sorted/eng_Latn.map | wget -x -nH --cut-dirs=2 -i -

In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes zstandard sentence-transformers faiss-cpu

### Load .zst Files

In [4]:
import zstandard as zstd
import json
import os

def read_zst(path):
    with open(path, 'rb') as f:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(f) as reader:
            text = reader.read().decode('utf-8', errors='ignore')
            for line in text.splitlines():
                try:
                    yield json.loads(line)
                except:
                    continue

### Load ALL datasets

In [5]:
base_path = "Data"

files = {
    "en": "eng_Latn/10_1.jsonl.zst",
    "sv": "swe_Latn/10_1.jsonl.zst",
    "it": "ita_Latn/10_1.jsonl.zst",
    "de": "deu_Latn/10_1.jsonl.zst",
    "fr": "fra_Latn/10_1.jsonl.zst",
}

data = []

for lang, path in files.items():
    full_path = os.path.join(base_path, path)
    print("Loading:", full_path)
    
    for row in read_zst(full_path):
        if "text" in row:
            data.append({
                "text": row["text"],
                "lang": lang
            })

print("Total samples:", len(data))

Loading: Data/eng_Latn/10_1.jsonl.zst
Loading: Data/swe_Latn/10_1.jsonl.zst
Loading: Data/ita_Latn/10_1.jsonl.zst
Loading: Data/deu_Latn/10_1.jsonl.zst
Loading: Data/fra_Latn/10_1.jsonl.zst
Total samples: 383381


### Clean dataset

In [6]:
def clean(text):
    if not text:
        return None
    text = text.strip()
    if len(text) < 5:
        return None
    return text

cleaned = []

for d in data:
    t = clean(d["text"])
    if t:
        cleaned.append({"text": t, "lang": d["lang"]})

data = cleaned

### Assign Experts

In [7]:
def assign_expert(lang):
    if lang == "en":
        return "expert_en"
    elif lang in ["de", "sv"]:
        return "expert_de_sv"
    elif lang in ["fr", "it"]:
        return "expert_fr_it"

for d in data:
    d["expert"] = assign_expert(d["lang"])

### Convert to HuggingFace Dataset

In [8]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset

Dataset({
    features: ['text', 'lang', 'expert'],
    num_rows: 383381
})

### Split per expert

In [9]:
dataset_en = dataset.filter(lambda x: x["expert"] == "expert_en")
dataset_desv = dataset.filter(lambda x: x["expert"] == "expert_de_sv")
dataset_frit = dataset.filter(lambda x: x["expert"] == "expert_fr_it")

Filter:   0%|          | 0/383381 [00:00<?, ? examples/s]

Filter:   0%|          | 0/383381 [00:00<?, ? examples/s]

Filter:   0%|          | 0/383381 [00:00<?, ? examples/s]

### Load Qwen2.5 Model (4-bit)

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# IMPORTANT
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

base_model.gradient_checkpointing_enable()

print("Base model loaded")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Base model loaded


### LoRA Setup

In [12]:
def create_lora_model(base_model):

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,

        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj"
        ],

        task_type=TaskType.CAUSAL_LM
    )

    model = get_peft_model(base_model, lora_config)

    model.print_trainable_parameters()

    return model

### Tokenization Function

In [35]:
def tokenize_function(batch):

    tokens = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    # IMPORTANT FOR LOSS
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

### Prepare Dataset

In [15]:
#English expert
tokenized_en = dataset_en.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset_en.column_names
)

print(tokenized_en)

Map:   0%|          | 0/149975 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 149975
})


In [16]:
# DE+SW
tokenized_desv = dataset_desv.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset_desv.column_names
)

Map:   0%|          | 0/92067 [00:00<?, ? examples/s]

In [17]:
# FR+IT
tokenized_frit = dataset_frit.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset_frit.column_names
)

Map:   0%|          | 0/141339 [00:00<?, ? examples/s]

### Data Collator

In [38]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

In [39]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

### Training Arguments

In [40]:
import wandb
print(wandb.run)

In [41]:
import wandb

wandb.init(
    project="HTYLLM2-Hybrid",
    name="expert-en-training"
)

train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
train/grad_norm,▃▁▇▃▄▂▂▂▂▄▂▃▂▂▁▄▂▁▃▄▂▁▂▃▄▄▃▂▂█▂▃▂▂▃▃▄▅▃▅
train/learning_rate,████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁
train/loss,▆▃▄▆█▅▄▆▆▇▄▅▅▃▅▄▇▃▅▅▁▄▅▄▅▃▅▅▄▃▆▆▅▅▇█▆▄▆█
train/epoch,0.10028
train/global_step,470
train/grad_norm,0.1247
train/learning_rate,0.00018
train/loss,2.71226


In [42]:
training_args = TrainingArguments(
    output_dir="temp_output",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    num_train_epochs=1,

    logging_steps=10,
    save_steps=500,

    bf16=True,

    report_to="wandb",

    remove_unused_columns=False
)

### Train EN Expert

In [43]:
model_en = create_lora_model(base_model)

trainer_en = Trainer(
    model=model_en,
    args=training_args,
    train_dataset=tokenized_en,
    data_collator=data_collator
)

trainer_en.train()

model_en.save_pretrained("experts/expert_en_lora")

print("EN expert saved")

/upb/users/a/ashaben/profiles/unix/cs/.local/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/upb/users/a/ashaben/profiles/unix/cs/.local/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


Step,Training Loss
10,2.654449
20,2.597792
30,2.624287
40,2.680449
50,2.722296
60,2.640097
70,2.704705
80,2.605148
90,2.665869
100,2.656383


EN expert saved


### Train DE+SV Expert

In [45]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

base_model.gradient_checkpointing_enable()

print("Base model loaded")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Base model loaded


In [46]:
import wandb

wandb.init(
    project="HTYLLM2-Hybrid",
    name="expert-desw-training"
)

train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▆▆▆▆▆▆▆▆▆▇▇████
train/grad_norm,▇▃▁▂▁▃▃▃▃▂▅▃▅▃▅▆▅▄▄▅▇▆▇█▆▃▅▅▆▆▇▆▆▆▆▅▅▅▅▆
train/learning_rate,██▇▇▇▇▆▆▆▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁
train/loss,▅▅▄▆▄▅▇█▆▅▄▄▄▇▄▂█▆▅▃▃▄▆▆▅▅▆▅▂▅▇▂▂▄▁▃▃▂▂▄
total_flos,3.262248143486976e+18
train/epoch,1
train/global_step,4687
train/grad_norm,0.13503
train/learning_rate,0.0
train/loss,2.65298


In [47]:
model_desv = create_lora_model(base_model)

trainer_desv = Trainer(
    model=model_desv,
    args=training_args,
    train_dataset=tokenized_desv,
    data_collator=data_collator
)

trainer_desv.train()

model_desv.save_pretrained("expert_desv_lora")

print("DE+SV expert saved")

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


Step,Training Loss
10,3.089867
20,3.080592
30,2.932481
40,2.788294
50,2.852317
60,2.906538
70,2.857544
80,2.779503
90,2.859883
100,2.901971


DE+SV expert saved


### Train FR+IT Expert

In [48]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

base_model.gradient_checkpointing_enable()

print("Base model loaded")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Base model loaded


In [49]:
import wandb

wandb.init(
    project="HTYLLM2-Hybrid",
    name="expert-frit-training"
)

train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇█
train/global_step,▁▁▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇███
train/grad_norm,▁▄▄▆█▆▅▅▅▅▇▆▅▇▆▆▇▆▇▅▆▆▇▅▇▇█▇▆▇▅▆▇███▆▇▆▆
train/learning_rate,███▇▇▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁
train/loss,█▅▆▃▄▄▃▃▃▃▂▃▂▂▃▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▂▁▂▂▂▂
total_flos,2.0026364382491443e+18
train/epoch,1
train/global_step,2878
train/grad_norm,0.42753
train/learning_rate,0.0
train/loss,2.4741


In [50]:
model_desv = create_lora_model(base_model)

trainer_desv = Trainer(
    model=model_desv,
    args=training_args,
    train_dataset=tokenized_desv,
    data_collator=data_collator
)

trainer_desv.train()

model_desv.save_pretrained("experts/expert_frit_lora")

print("FR+IT expert saved")

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


Step,Training Loss
10,3.089812
20,3.080761
30,2.932560
40,2.788248
50,2.852452
60,2.906511
70,2.857539
80,2.779631
90,2.859792
100,2.901929


FR+IT expert saved


### Build Embedding Router (BGE-M3)

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [2]:
embedder = SentenceTransformer(
    "BAAI/bge-m3",
    device="cuda"
)

print("Embedder loaded")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedder loaded


### Build Routing Vectors

In [12]:
import os

print(os.listdir("Data/processed"))

['.ipynb_checkpoints', 'expert_fr_it.jsonl', 'expert_de_sv.jsonl', 'expert_en.jsonl']


In [3]:
from datasets import load_dataset


dataset_en = load_dataset(
    "json",
    data_files="Data/processed/expert_en.jsonl",
    split="train"
)

print("dataset en loaded")

dataset_desv = load_dataset(
    "json",
    data_files="Data/processed/expert_de_sv.jsonl",
    split="train"
)

print("dataset desv loaded")


dataset_frit = load_dataset(
    "json",
    data_files="Data/processed/expert_fr_it.jsonl",
    split="train"
)

print("dataset frit loaded")



dataset en loaded
dataset desv loaded
dataset frit loaded


In [5]:
import gc
import torch
import numpy as np

expert_vectors = {}

def build_embeddings(dataset, key, batch_size=16):
    texts = dataset.select(range(5000))["text"]

    with torch.no_grad():
        embs = embedder.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,   # VERY IMPORTANT
        )

    expert_vectors[key] = embs.astype(np.float32)

    gc.collect()
    torch.cuda.empty_cache()

    print(f"{key} done")


build_embeddings(dataset_en, "expert_en")
build_embeddings(dataset_desv, "expert_de_sv")
build_embeddings(dataset_frit, "expert_fr_it")

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

expert_en done


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

expert_de_sv done


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

expert_fr_it done


### Router Function

In [6]:
def route(text):
    emb = embedder.encode(text)

    best_expert = None
    best_score = -1

    for expert, centroid in centroids.items():
        score = np.dot(emb, centroid) / (
            np.linalg.norm(emb) * np.linalg.norm(centroid)
        )

        if score > best_score:
            best_score = score
            best_expert = expert

    return best_expert

### Load Experts for Inference

In [8]:
from transformers import AutoModelForCausalLM

In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
)

# Load first adapter
model = PeftModel.from_pretrained(
    base_model,
    "experts/expert_en_lora",
    adapter_name="en"
)

model.load_adapter("experts/expert_desv_lora", adapter_name="desv")
model.load_adapter("experts/expert_frit_lora", adapter_name="frit")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

<All keys matched successfully>

### Full Inference Pipeline

In [30]:
import numpy as np

In [47]:
centroids = {
    k: np.mean(v, axis=0)
    for k, v in expert_vectors.items()
}

In [48]:
experts = {
    "expert_en": "en",
    "expert_de_sv": "desv",
    "expert_fr_it": "frit"
}

In [56]:
def route(text):
    emb = embedder.encode(text, normalize_embeddings=True)

    best_expert = max(
        centroids.items(),
        key=lambda x: np.dot(emb, x[1])
    )[0]

    return best_expert

In [57]:
def generate(text):
    expert_id = route(text)

    adapter = experts[expert_id]   # <-- FIXED

    model.set_adapter(adapter)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7
    )
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # log to wandb
    wandb.log({
        "input_text": text,
        "selected_expert": expert_id,
        "adapter_used": adapter,
        "output_text": decoded
    })

    return decoded



### Test

In [58]:
print(generate("How are you today?"))
print(generate("Wie geht es dir?"))
print(generate("Comment ça va?"))

How are you today? Are you having a good day? Have you been busy? Well, it’s the weekend and you should be having fun. So why are you sitting here in front of your computer reading this blog post? I’m guessing that you’re looking for something to do. Or maybe you just need some inspiration.
Well, I have a great idea for you. Let’s go on a hike! It’s a great way to get out of the house, enjoy the fresh air, and get some exercise. Plus, it’s a great way to spend time with friends or family. So grab your hiking boots and let’s hit the trails!
Hiking is a great way to get out of the house and enjoy the great outdoors. It’s also a great way to get some exercise and fresh air. Plus, it’s a great way to spend time with friends or family.
There are many different types of hikes available, so you can find one that fits your interests and abilities. You can go on a
Wie geht es dir? Ich hoffe du bist gut in deinem Leben und hast Spaß an deiner Zeit. Ich bin ein 27jähriger Mann mit einem großartig